# DhikrSpeech — one model per dhikr, end to end

From the recordings on Drive to the quantised TFLite model the Android app loads for **one**
dhikr. The production architecture is not a classifier: the user picks a dhikr before a
listening session, and the app loads that dhikr's own model, whose only job is

| | |
|---|---|
| **TARGET** | the selected phrase, spoken completely |
| **UNKNOWN** | everything else — other dhikr, *incomplete versions of this phrase*, ordinary speech, Quran, TV, noise, silence |

so training here is **binary phrase spotting**, and a false count is far more damaging than an
occasional miss.

With `target.phrase_id: all`, run the **batch build** cell after Setup to train and export every
phrase as an independent model. The detailed stages below remain a drill-down workflow for one
chosen `TARGET_PHRASE_ID` when you need to inspect charts or diagnose that target.

1. **Dataset** — what this target actually has: positives, negatives by category, speakers, window length.
2. **Preprocessing** — condition the audio, split by **speaker**, write the manifest.
3. **Training** — one detector, with the negative pool sampled down to a workable ratio.
4. **Evaluation** — clip metrics per *negative category*, not one accuracy.
5. **Export** — TFLite variants, INT8 verification, the Android metadata contract.
6. **Streaming** — the stage that decides shipping: events, FA/hour, threshold calibration, readiness.
7. **Architectures** — DS-CNN Tiny vs DS-CNN vs TC-ResNet8 on the same split (optional).
8. **Experiment** — the older "one model per dhikr?" comparison, kept for the record (optional).

Stages hand off through files on Drive, so a stage can be re-run on its own after Setup.

**Runtime → Change runtime type → GPU** before training.

> Stage 6 is not optional polish. Clip accuracy cannot tell you whether a detector counts one
> event per repetition on continuous audio, and the readiness verdict refuses to say
> `READY FOR DEVICE TEST` without it.


In [26]:
#@title Setup — mount Drive, locate the project, install what is missing
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MahmoudMabrok/SaloAleh.git"
REPO_BRANCH = os.environ.get("DHIKR_BRANCH", "main")
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def sync_clone(target: Path) -> None:
    """Pull the latest code into an existing clone.

    Without this a runtime that cloned the repository earlier keeps running that
    old copy for the rest of the session, so fixes never arrive.
    """
    subprocess.run(
        ["git", "-C", str(target), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True
    )
    subprocess.run(
        ["git", "-C", str(target), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"],
        check=True,
    )


def find_project_root() -> Path:
    """The folder that holds src/ and configs/ — cloned or updated as needed."""
    candidates = [Path(p) for p in [
        os.environ.get("DHIKR_PROJECT_ROOT", ""),
        "/content/DhikrSpeech",
        "/content/SaloAleh/DhikrSpeech",
        "/content/drive/MyDrive/DhikrSpeech",
    ] if p]
    candidates += [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src" / "config.py").is_file() and (candidate / "configs" / "config.yaml").is_file():
            # Only ever update the throwaway clone this notebook created. A repo
            # you checked out yourself is left alone - resetting it would discard
            # whatever branch and local edits you are working on.
            if IN_COLAB and candidate.parent == Path("/content/SaloAleh"):
                try:
                    sync_clone(candidate.parent)
                except Exception as error:
                    print("could not update the clone, using it as is:", error)
            return candidate.resolve()
    if IN_COLAB:
        target = Path("/content/SaloAleh")
        print("cloning", REPO_URL, "@", REPO_BRANCH)
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(target)],
            check=True,
        )
        return (target / "DhikrSpeech").resolve()
    raise FileNotFoundError(
        "DhikrSpeech project not found. Set DHIKR_PROJECT_ROOT, or copy the "
        "DhikrSpeech folder to /content or to your Drive."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# A kernel that already imported src/ keeps the old modules even after the clone
# is updated, so drop them and let the imports below load the new code.
stale = [name for name in list(sys.modules) if name == "src" or name.startswith("src.")]
for name in stale:
    del sys.modules[name]

for module_name, package in [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("yaml", "PyYAML"),
    ("sklearn", "scikit-learn"),
    ("soxr", "soxr"),
]:
    if importlib.util.find_spec(module_name) is None:
        print("installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

from src.config import load_config

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
config = load_config(CONFIG_PATH)
config.paths.ensure_dirs()

revision = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip()

print("project root :", PROJECT_ROOT)
print("code version :", revision or "(not a git checkout)")
print("config       :", CONFIG_PATH)
if stale:
    print("note         : reloaded %d cached src modules — re-run this notebook "
          "from the top so every stage uses the new code" % len(stale))
print()
print(config.summary())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


INFO src.config: loaded configuration from /content/SaloAleh/DhikrSpeech/configs/config.yaml


project root : /content/SaloAleh/DhikrSpeech
code version : 6856649 djir modle
config       : /content/SaloAleh/DhikrSpeech/configs/config.yaml
note         : reloaded 9 cached src modules — re-run this notebook from the top so every stage uses the new code

project root      : /content/drive/MyDrive/Dhikr Speech Dataset
classes           : phrases [6, 7] only
sample rate       : 16000 Hz, mono, PCM16
clip length       : 2 s (32000 samples)
features          : 40 log-mel bins x 197 frames (window 30 ms / hop 10 ms)
model             : ds_cnn (4 blocks x 64 filters)
training          : 300 epochs, batch 16, optimizer adam
augmentation      : on
seed              : 1337


## Batch build — one model for every phrase

Run the next cell **instead of the detailed cells below** when the goal is to build the full model
set. It resolves `target.phrase_id: all` to numeric dataset folders also present in
`phrases.json`, then runs dataset → training → evaluation → streaming calibration → export
separately for every id. Catalog entries without recordings are skipped; a failed collected phrase is reported
without discarding the models completed for the others.


In [ ]:
#@title Build every configured phrase model end to end
BUILD_ALL_PHRASE_MODELS = config.target.batch_enabled

if BUILD_ALL_PHRASE_MODELS:
    command = [
        sys.executable, str(PROJECT_ROOT / "train.py"),
        "--config", str(CONFIG_PATH), "--all-targets",
    ]
    print("running:", " ".join(command))
    completed = subprocess.run(command)
    if completed.returncode:
        print("\nBATCH INCOMPLETE: completed target exports were kept.")
        print("The final 'failed targets' table above contains each actionable error.")
        print("Run the failed id alone with: python train.py --target <id>")
    else:
        print("all phrase exports:", config.paths.exports_path)
else:
    print("Batch mode is off. Set target.phrase_id: all in configs/config.yaml, rerun Setup, "
          "then run this cell again.")


## Choose one target to inspect

One id, one model. Everything below — the manifest, the checkpoints, the export folder — is
scoped to it. This is the drill-down path; use the batch cell above to build every phrase.

`config.for_target()` also applies any `target.phrase_overrides` for this id: today that is
`clip_seconds`, because a two-word dhikr and a seven-word one do not belong in the same window.


In [ ]:
from src.dataset import load_phrases

TARGET_PHRASE_ID = 7  #@param {type:"integer"}  # e.g. 7 -> dataset/007
config = config.for_target(TARGET_PHRASE_ID)
paths = config.paths

phrases = load_phrases(paths.phrases_path)
TARGET_TEXT = next((p.text for p in phrases if p.id == TARGET_PHRASE_ID), "")

print("TARGET     :", config.target.folder, TARGET_TEXT)
print("output     :", config.target.output_mode, f"({config.target.num_outputs} output(s))")
print("window     :", f"{config.audio.clip_seconds:g} s ({config.audio.clip_samples} samples)")
print("clip cache :", config.clip_cache_path())
print("manifest   :", config.target_manifest_path())
print("exports    :", config.target_export_path())
print()
print(config.summary())


# 01 · Dataset

Three questions before a single epoch is trained, and none of them is "what is the accuracy":

* how many recordings of this phrase, **from how many different speakers**;
* what the negatives are made of — and whether any of them are *hard*, i.e. near-misses of this
  exact phrase (`سبحان الله`, `سبحان الله العظيم` for target 007);
* whether the window is long enough to hold the phrase with margin.

A cropped positive is indistinguishable from a partial phrase, which is precisely the
distinction the model has to learn.


## 1 · Locate the dataset

Paths come from `configs/config.yaml`. The layout, all optional except the target folder:

```
dataset/007/**                      the target                 -> TARGET
dataset/006/**  dataset/001/**      other dhikr                -> other_dhikr
dataset/unknown/*.wav               flat filler                -> unknown
  *_hard_negative_006.wav           hard negative for 006 only -> hard_negative
dataset/unknown/normal_speech/      ordinary Arabic speech     -> normal_speech
dataset/unknown/hard_negative/      near-misses of a phrase    -> hard_negative
dataset/unknown/partial_phrase/     incomplete utterances      -> partial_phrase
dataset/unknown/other_dhikr/        recorded as negatives      -> other_dhikr
dataset/unknown/noise/              room, street, TV           -> noise
```

The category is the first subfolder under `unknown/`. Everything below it trains as one
`UNKNOWN` class — the subfolder never becomes an output — but it is *evaluated* separately, which
is what says which kind of audio breaks the detector.

A filename ending in `_hard_negative_<target_id>` is target-scoped. For example,
`unknown_spABC_01_000_hard_negative_006.wav` is used as a hard negative only while building
target `006`; it is excluded from every other target so it cannot teach another model to reject
its own phrase. Untagged files under `unknown/hard_negative/` remain shared by every target.


In [ ]:
from pathlib import Path
from src.targets import target_negative_paths

negative_paths = target_negative_paths(
    paths.dataset_path, config.target, unknown_class=paths.unknown_class
)

rows = [
    ("dataset", paths.dataset_path),
    ("  target", paths.dataset_path / (config.target.folder or "")),
    *[(f"  negative/{path.name}", path) for path in negative_paths],
    ("    hard negatives", paths.dataset_path / paths.unknown_class / "hard_negative"),
    ("phrases.json", paths.phrases_path),
    ("processed", paths.processed_path),
    ("streaming set", config.paths.resolve_streaming_path()),
    ("checkpoints", paths.checkpoints_path),
    ("exports", config.target_export_path()),
    ("reports", paths.reports_path),
    ("noise (optional)", paths.noise_path),
]
for name, path in rows:
    print("%-18s %-8s %s" % (name, "ok" if Path(path).exists() else "MISSING", path))

# The streaming set gets its own line: "the folder is missing", "the folder is
# there but has no annotations.json" and "annotated and ready" need different
# answers, and reporting MISSING for all three sends you looking for a folder
# that is already on Drive. It also finds the set when it was filed under
# a name other than the configured `paths.streaming_dir`.
from src.streaming_eval import streaming_status

print()
print("streaming set     :", streaming_status(config))

if not paths.dataset_path.is_dir():
    raise FileNotFoundError(
        "dataset folder not found: %s\nPoint paths.drive_root / paths.project_dir at the "
        "right place in configs/config.yaml." % paths.dataset_path
    )


## 2 · Index this target

Every clip is mapped to TARGET or UNKNOWN, and every negative keeps the *category* it came
from. They all train as UNKNOWN — the categories exist so that evaluation can say **what**
produces a false detection, which is the difference between "the model is 99% accurate" and
"the model cannot tell a complete phrase from its first three words".


In [ ]:
import pandas as pd

from src.targets import NEGATIVE_TYPES, negative_breakdown, scan_target_dataset

index = scan_target_dataset(
    paths.dataset_path,
    phrases,
    config.target,
    unknown_class=paths.unknown_class,
    extensions=config.audio.file_extensions,
)

positives = sum(1 for s in index.samples if s.negative_type is None)
breakdown = negative_breakdown(index.samples)
summary = pd.DataFrame(
    [{"class": "TARGET", "category": config.target.folder, "clips": positives}]
    + [
        {"class": "UNKNOWN", "category": name, "clips": breakdown.get(name, 0)}
        for name in NEGATIVE_TYPES
        if breakdown.get(name)
    ]
)
print("target :", config.target.folder, index.target_text)
print("clips  :", len(index.samples), f"({positives} target / {len(index.samples) - positives} other)")
summary


## 3 · Validate

Every file is opened and decoded — the slow cell. It finds the problems no amount of training
recovers from: unreadable files, silence, duplicates, wrong sample rates.


In [ ]:
from src.dataset import validate_dataset

DEEP = True   # False = headers only (fast), no silence or duplicate detection

report = validate_dataset(index, config.audio, deep=DEEP)
print(report.summary())


In [ ]:
issues = report.issues_dataframe()
if len(issues):
    display(issues.groupby("kind").size().rename("count").to_frame())
    display(issues.head(20))
else:
    print("no issues found.")

blocked = report.unusable_paths()
print("\nfiles that will be excluded from preprocessing:", len(blocked))


## 4 · Speakers

The number that matters most, and the one nobody tracks. A model that heard the same voice in
training and in validation reports an accuracy that says nothing about a stranger's phone.

Three sources, tried in order by `split.speaker.source: auto` — `speakers.csv`, a per-speaker
subfolder, then the filename patterns. SpeechCollector names its uploads
`<class>_sp<8 hex>_<timestamp>_<suffix>`, so the device token is matched **first**: a pattern that
simply took the leading token would extract the *class id* and make every recording of a phrase
one "speaker".

If none resolves, this prints the warning in full, because every number downstream is then
optimistic and should be read that way. `src/speaker_backfill.py` can give pre-token recordings a
derived token from the collector's metadata sheet.


In [ ]:
from src.dataset import build_speaker_resolver
from src.speakers import speaker_report

resolver = build_speaker_resolver(config)
index = scan_target_dataset(
    paths.dataset_path,
    phrases,
    config.target,
    unknown_class=paths.unknown_class,
    extensions=config.audio.file_extensions,
    speaker_resolver=resolver,
)

report_speakers = speaker_report(
    [(s.speaker, s.label, "") for s in index.samples], index.speaker_source
)
resolved = [s for s in index.samples if s.speaker]
target_speakers = {s.speaker for s in index.samples if s.negative_type is None and s.speaker}
print("speaker source               :", index.speaker_source)
print("recordings with a speaker id :", len(resolved), "of", len(index.samples))
print("distinct speakers            :", report_speakers.num_speakers)
print("speakers who said the target :", len(target_speakers))
if not resolved:
    print()
    print("!! EVALUATION IS NOT SPEAKER-INDEPENDENT.")
    print("   Add speakers.csv, or backfill device tokens onto pre-token recordings")
    print("   with src/speaker_backfill.py. Training still works; the numbers just do not")
    print("   describe how the model behaves on a voice it has never heard.")


## 5 · The dataset report

Counts, durations, speakers, the negative breakdown, and a **recommended window length** from
the measured utterance durations. The recommendation is reported, never applied: changing
`audio.clip_seconds` invalidates every cached clip and every checkpoint, so it stays a decision
somebody makes.

The notes at the bottom are ordered by how much they cost. They do not block anything — a
40-clip prototype is a legitimate thing to run, it just must not be reported as a result.


In [ ]:
from src.targets import build_target_report

durations = {stat.path: stat.duration for stat in report.file_stats if stat.ok}
dataset_report = build_target_report(index, durations=durations, config=config)
print(dataset_report.summary())


In [ ]:
from src import visualization as viz

stats = report.stats
figure = viz.plot_duration_histogram(
    [stat.duration for stat in report.file_stats if stat.ok and stat.duration > 0],
    min_duration=config.audio.min_duration,
    max_duration=config.audio.max_duration,
    title=f"recording duration — target {config.target.folder} and its negatives",
)
display(figure)

counts = {"TARGET": dataset_report.positive_clips}
counts.update({name: int(entry["clips"]) for name, entry in dataset_report.by_negative_type.items()})
display(viz.plot_class_distribution(counts, title="clips per class / negative category"))


## 6 · Listen

The check no validator performs: does the target folder actually contain the target phrase, and
do the hard negatives actually sound like near-misses of it? A mislabelled folder trains
perfectly and fails on the phone.


In [ ]:
import numpy as np
from IPython.display import Audio, display

from src.audio import load_audio

groups = {"TARGET": [s for s in index.samples if s.negative_type is None]}
for name in ("hard_negative", "partial_phrase", "other_dhikr", "general_speech", "noise"):
    picked = [s for s in index.samples if s.negative_type == name]
    if picked:
        groups[name] = picked

rng = np.random.default_rng(config.seed)
for name, samples in groups.items():
    sample = samples[int(rng.integers(len(samples)))]
    print(f"{name:<16} {sample.path.name}")
    display(Audio(load_audio(sample.path, config.audio.sample_rate), rate=config.audio.sample_rate))


In [ ]:
written = report.save(paths.reports_path, name=f"validation_target_{config.target.folder}")
for kind, path in written.items():
    print("%-6s %s" % (kind, path))

import json

destination = paths.reports_path / f"dataset_target_{config.target.folder}.json"
destination.write_text(json.dumps(dataset_report.to_dict(), indent=2, ensure_ascii=False), encoding="utf-8")
print("report", destination)


# 02 · Preprocessing

Conditions every recording to the exact tensor the model will see, splits **by speaker**, and
writes this target's manifest.

Two things are worth knowing about where the files go:

* clips are cached under `processed/audio/<sample rate>hz_<window>s/<source folder>/`, so the
  shared negative pool is conditioned **once** and reused by every target with the same window
  geometry — training a second target does not re-write the whole pool;
* a target that overrides `clip_seconds` gets its own cache, rather than silently reusing clips
  fitted to a different length.


## 1 · Condition, split, write the manifest

`prepare_target` runs the whole stage. The split is by speaker and is **verified afterwards**,
not assumed: if a voice ends up in two splits the call raises rather than handing back a
manifest whose numbers cannot be trusted (`split.fail_on_leakage`).


In [ ]:
from src.pipeline import prepare_target

OVERWRITE_AUDIO = False   # True after changing any audio.* setting

preparation = prepare_target(config, TARGET_PHRASE_ID, overwrite=OVERWRITE_AUDIO)
records = preparation.records
print(preparation.summary())


## 2 · Verify the split

Explicitly, because leakage is silent — it does not fail anything, it just inflates every
number the run reports.


In [ ]:
import pandas as pd

from src.dataset import split_counts
from src.targets import TARGET_INDEX

print(preparation.speakers.summary())
print()

rows = []
for split in ("train", "val", "test"):
    subset = [r for r in records if r.split == split]
    rows.append({
        "split": split,
        "clips": len(subset),
        "target": sum(1 for r in subset if r.class_index == TARGET_INDEX),
        "unknown": sum(1 for r in subset if r.class_index != TARGET_INDEX),
        "speakers": len({r.speaker for r in subset if r.speaker}),
    })
display(pd.DataFrame(rows))

by_type = (
    pd.DataFrame([
        {"split": r.split, "negative_type": r.negative_type or "target"}
        for r in records
    ])
    .value_counts()
    .unstack(fill_value=0)
)
display(by_type)


## 3 · Negative sampling

The negative pool is meant to grow without bound while the positives stay in the hundreds, so
training on all of it every run drowns the phrase. `negative_sampling.ratio` caps negatives at
`ratio × positives`; the weights decide *which* survive the cut — a hard negative is worth
several clips of room tone.

Sampling is without replacement and deterministic in the seed, and what was dropped is printed
rather than silently discarded.


In [ ]:
from src.pipeline import training_records

train_records = training_records(config, records)
available = [r for r in records if r.split == "train"]

before = pd.Series([r.negative_type or "target" for r in available]).value_counts()
after = pd.Series([r.negative_type or "target" for r in train_records]).value_counts()
comparison = pd.DataFrame({"available": before, "sampled": after}).fillna(0).astype(int)
comparison["kept"] = (comparison["sampled"] / comparison["available"].replace(0, 1)).round(2)
display(comparison)

positives = sum(1 for r in train_records if r.class_index == TARGET_INDEX)
print("training clips :", len(train_records))
print("positives      :", positives)
print("negatives      :", len(train_records) - positives,
      f"({(len(train_records) - positives) / max(positives, 1):.1f}x positives)")


## 4 · What the model sees

The front-end, on one target clip and one hard negative. If the target and its near-miss look
identical here, no architecture will separate them — that is a window-length or a data problem.


In [ ]:
import numpy as np
from IPython.display import Audio, display

from src.audio import read_wav
from src.features import LogMelExtractor
from src import visualization as viz

extractor = LogMelExtractor(config=config.features, sample_rate=config.audio.sample_rate)
root = config.clip_cache_path()

picks = []
target_clip = next((r for r in records if r.class_index == TARGET_INDEX), None)
hard_clip = next((r for r in records if r.negative_type == "hard_negative"), None)
other_clip = next((r for r in records if r.negative_type == "other_dhikr"), None)
for label, record in (("TARGET", target_clip), ("hard negative", hard_clip), ("other dhikr", other_clip)):
    if record is not None:
        picks.append((label, record))

features, titles = [], []
for label, record in picks:
    samples, _ = read_wav(record.resolve(root))
    features.append(extractor(samples))
    titles.append(f"{label} — {record.path}")
    print(f"{label:<14} {record.path}")
    display(Audio(samples, rate=config.audio.sample_rate))

print("feature shape :", features[0].shape, "(frames, mel bins)")
display(viz.plot_feature_grid(features, titles, hop_ms=config.features.hop_ms, columns=len(features)))


## 5 · Augmentation, and whether the noise is real

Augmentation runs on the fly during training only. The one thing to check here is the line this
cell prints: **REAL NOISE** or **SYNTHETIC FALLBACK**. White and pink noise teach robustness to
a sound no phone ever hears; a folder of real room, street and TV recordings under
`paths.noise_dir` is worth considerably more, and it is the same material the negative pool
wants anyway.


In [ ]:
from src.augmentation import NoiseBank, WaveformAugmentor, spec_augment
from src.pipeline import describe_noise

noise_bank = NoiseBank.load(
    paths.noise_path,
    config.audio.sample_rate,
    allow_synthetic=config.augmentation.background_noise.synthetic_when_missing,
)
print("background noise:", describe_noise(noise_bank))
print()

augmentor = WaveformAugmentor(config=config.augmentation, audio=config.audio, noise_bank=noise_bank)
samples, _ = read_wav(target_clip.resolve(root))
rng = np.random.default_rng(config.seed)

variants = [("original", extractor(samples))]
for index in range(3):
    augmented = augmentor(samples, rng)
    variants.append((f"augmented {index + 1}", spec_augment(extractor(augmented), config.augmentation, rng)))

display(Audio(augmentor(samples, np.random.default_rng(config.seed)), rate=config.audio.sample_rate))
display(viz.plot_feature_grid([f for _, f in variants], [t for t, _ in variants],
                              hop_ms=config.features.hop_ms, columns=4))


# 03 · Training

One detector, `target.output_mode` outputs (2 for softmax, 1 for sigmoid). Class weights
counteract the negative-heavy split, so a model that never fires is not rewarded for it.

`model.name` picks the architecture — `ds_cnn` (baseline), `ds_cnn_tiny`, `tc_resnet8` — and
`model_presets` supplies only the values that make it that architecture, so switching changes
the architecture and nothing else.


## 1 · Seed, precision, device


In [ ]:
import tensorflow as tf

from src.trainer import configure_mixed_precision, set_global_seed

set_global_seed(config.seed)
mixed = configure_mixed_precision(config.training.mixed_precision)

devices = tf.config.list_physical_devices("GPU")
print("tensorflow      :", tf.__version__)
print("gpu             :", devices or "none - training will be slow")
print("mixed precision :", "on" if mixed else "off")


## 2 · The model

Check the parameter count against the number of training clips. A network with more parameters
than it has recordings will drive training accuracy to 1.0 while validation plateaus; the fix
order that actually works is *more speakers → more recordings → less capacity → more
augmentation*, and `ds_cnn_tiny` is the "less capacity" step made explicit.


In [ ]:
from src.models import build_model, estimate_flops, model_summary_text

ARCHITECTURE = config.model.name   # ds_cnn | ds_cnn_tiny | tc_resnet8
config = config.with_overrides({"model.name": ARCHITECTURE})

model = build_model(config.input_shape, config.target.num_outputs, config.resolved_model())
model.summary()
print("\nparameters :", f"{model.count_params():,}")
print("FLOPs      :", f"{estimate_flops(model):,}")
print("train clips:", len(train_records))


## 2b · Sanity check — can this pipeline learn at all?

Before spending an hour, prove the model can **memorise a handful of clips**. It only has to
overfit them; a correct pipeline reaches near-100% in a couple of hundred steps. If this sits at
chance, the fault is upstream of every hyperparameter — features, labels, or the topology.


In [ ]:
from src.dataset import make_tf_dataset
from src.trainer import sanity_overfit, sanity_overfit_report

SANITY_CLIPS = 40
SANITY_STEPS = 200

subset = train_records[:SANITY_CLIPS]
sanity_dataset = make_tf_dataset(
    subset, config, extractor, training=False, processed_root=root,
    batch_size=8, shuffle=False,
)
result = sanity_overfit(model, sanity_dataset, steps=SANITY_STEPS)
print(sanity_overfit_report(result, num_classes=max(config.target.num_outputs, 2)))


## 3 · Train

Checkpoints, logs and a config snapshot are written to Drive as training runs, so an
interrupted session resumes. `FRESH_START = True` after **any** config change — a resumed run
applies the new settings on top of the old weights and quietly trains something the config no
longer describes.


In [ ]:
RUN_NAME = None    # None -> target_<id>_<architecture>
LOG_DIR = str(paths.logs_path / (RUN_NAME or f"target_{config.target.folder}_{ARCHITECTURE}"))
print("log dir:", LOG_DIR)

%load_ext tensorboard
%tensorboard --logdir "$LOG_DIR"


In [ ]:
from src.pipeline import train_target

FRESH_START = False

trainer, artifacts = train_target(
    preparation,
    run_name=RUN_NAME,
    fresh=FRESH_START,
    architecture=ARCHITECTURE,
    verbose=1,
)
print()
print(artifacts.summary())


In [ ]:
display(viz.plot_training_history(artifacts.history, title=f"target {config.target.folder} — {ARCHITECTURE}"))


# 04 · Evaluation (clips)

Accuracy is deliberately not the headline. With negatives outnumbering positives 2:1 a model
that never fires is already 67% accurate, so what is reported is precision, recall, and the
false-positive rate **per negative category**.

That breakdown is the actionable part: a detector that is flawless on room tone and fires on
half the near-misses has one number worth knowing, and it is not its accuracy. It also names the
recordings to go and collect.

This stage still cannot tell you whether the model counts correctly — that is stage 6.


In [ ]:
from src.pipeline import evaluate_target

clip_evaluation = evaluate_target(preparation, trainer.load_best())
print(clip_evaluation.summary())


In [ ]:
display(viz.plot_detector_scores(
    clip_evaluation.scores,
    clip_evaluation.y_true,
    clip_evaluation.negative_types,
    threshold=clip_evaluation.threshold,
    title=f"P(target) by clip type — target {config.target.folder}",
))

breakdown = pd.DataFrame(clip_evaluation.by_negative_type()).T
if len(breakdown):
    display(breakdown.sort_values("false_positive_rate", ascending=False))


## Listen to the false positives

The fastest way to tell a model problem from a data problem. If a clip the model accepted really
does contain the target phrase, the label is wrong; if it is a near-miss, the model has learned
the opening words rather than the whole phrase, and the answer is more hard negatives.


In [ ]:
from IPython.display import Audio, display

for row in clip_evaluation.worst_false_positives(limit=6):
    print(f"P(target) {row['score']:.3f}  {row['negative_type']:<16} {row['path']}")
    samples, _ = read_wav(root / row["path"])
    display(Audio(samples, rate=config.audio.sample_rate))


In [ ]:
written = clip_evaluation.save(paths.reports_path / f"evaluation_target_{config.target.folder}.json")
print("saved:", written)


# 05 · Streaming — the stage that decides shipping

Everything so far scored *clips*. Production is an open microphone, and that changes the
problem twice over:

* a phrase passes through many overlapping windows, so `P(target) > threshold` counts one dhikr
  four or five times — and a plain refractory timer is no better, because any timer long enough
  to swallow a 2-second phrase also swallows the next repetition of someone saying it quickly;
* the model spends almost all of its time listening to things that are *not* the target, so the
  number that decides whether this is shippable is **false activations per hour**.

Counting is therefore a state machine with hysteresis, re-armed by the score *falling away*
rather than by a cooldown:

```
IDLE --score>=activation--> CANDIDATE --enough hits--> CONFIRMED
  ^                                                        |
  |                                          score<release for
  |                                          release_windows
  +---------------- cooldown elapsed ------------------ COOLDOWN
```

### What this stage needs

```
streaming/
  audio/session_001.wav        someone repeating the dhikr, minutes at a time
  audio/tv_arabic.wav          zero target phrases, shared by every target
  audio/stream_negative_006.wav zero target phrases, used only for target 006
  annotations.json
```

```json
[
  {"file": "session_001.wav", "target": "007",
   "events": [{"start": 12.3, "end": 14.1}, {"start": 19.0, "end": 20.8}]},
  {"file": "tv_arabic.wav", "target": "007", "category": "background_audio", "events": []}
]
```

A recording tagged `stream_negative_<target_id>` is automatically treated as a zero-count hard
negative for that target only; no manual annotation entry is required. A recording with **no**
events is a negative-only stress test: every event detected in it is a
false activation, and `category` says which kind of audio produced it. Those recordings are the
cheapest thing in this whole project to collect — leave a phone recording the television — and
they carry the single most important number.


## 1 · Score the long-form recordings

The model runs once per recording; every threshold in the sweep below then replays only the
state machine over the cached timelines. A 60-point sweep costs one forward pass, not sixty.


In [ ]:
from src.streaming import StreamingDetector
from src.streaming_eval import load_streaming_set, score_clips, streaming_audio_root

model = trainer.load_best()
audio_root = streaming_audio_root(config)
clips = load_streaming_set(config)
detector = StreamingDetector.from_config(config, model)

scored = score_clips(
    detector, clips, audio_root,
    target_folder=config.target.folder,
    progress=lambda name, position, total: print(f"  [{position}/{total}] {name}"),
)

annotated = sum(len(item.clip.events) for item in scored)
audio_hours = sum(item.timeline.duration_seconds for item in scored) / 3600.0
print()
print("recordings :", len(scored))
print("repetitions:", annotated)
print("audio      :", f"{audio_hours:.2f} h")
if not scored:
    print()
    print("!! No streaming recordings. This stage cannot run, and without it clip accuracy is")
    print("   all you have - which cannot tell you whether the detector counts once per")
    print("   repetition. The readiness verdict below will say EXPERIMENTAL for that reason.")


## 2 · Calibrate the threshold

`0.5` is a default, not a threshold. The activation threshold that ships is the **lowest** one
whose measured FA/hour stays inside the budget — lower thresholds detect more, so the lowest
admissible one is also the highest-recall admissible one.

When no threshold qualifies, this reports a **failure**. Reaching for 0.99 and calling it
calibrated would ship a model that counts almost nothing and hide the fact that the negatives,
not the threshold, are the problem.


In [ ]:
from src.streaming import detector_with_threshold
from src.streaming_eval import calibrate_threshold, evaluate_timelines

calibration = None
operating_point = config.streaming.detector

if scored and config.calibration.enabled:
    calibration = calibrate_threshold(
        scored, config.calibration, operating_point, config.streaming.match_tolerance_seconds
    )
    print(calibration.summary())
    if calibration.satisfied:
        operating_point = detector_with_threshold(
            operating_point, calibration.activation,
            release=calibration.release, release_ratio=config.calibration.release_ratio,
        )
    display(viz.plot_threshold_sweep(
        calibration.rows,
        budget=config.calibration.target_false_activations_per_hour,
        chosen=calibration.activation,
        title=f"threshold sweep — target {config.target.folder}",
    ))
    display(calibration.to_dataframe().head(30))


## 3 · Event metrics

Expected repetitions against detected ones. Note that **duplicates are counted separately from
false positives**: a duplicate is one utterance counted twice, which is the exact failure the
state machine exists to prevent, and folding it into "false positives" would hide it. It still
costs precision, because on the phone a duplicate *is* an extra count.


In [ ]:
streaming_evaluation = None
if scored:
    streaming_evaluation = evaluate_timelines(
        scored, operating_point, config.streaming.match_tolerance_seconds
    )
    print(streaming_evaluation.summary())


## 4 · The timeline

The most useful picture in the project. A missed repetition, a phrase counted twice and a false
activation on the television all look completely different here — and identical in a table of
accuracies.


In [ ]:
for item, result in list(zip(scored, streaming_evaluation.per_clip if streaming_evaluation else [])):
    display(viz.plot_score_timeline(
        item.timeline,
        events=result.events,
        truth=item.clip.events,
        activation=operating_point.activation_threshold,
        release=operating_point.release_threshold,
        title=f"{item.clip.file} — expected {result.metrics.expected}, "
              f"detected {result.metrics.detected}, false {result.metrics.false_events}",
    ))


## 5 · Negative-only stress test

Hours of audio containing **zero** target phrases: conversation, TV, Quran recitation, other
dhikr, street, room tone. Report total duration, false activations per hour, the peak confidence
reached, and which category produced each one.

This is release-critical, and it is the cheapest data in the project to collect.


In [ ]:
if streaming_evaluation:
    negative = streaming_evaluation.negative_only
    if negative.duration_seconds:
        print("duration          :", f"{negative.duration_seconds / 3600.0:.2f} h with no target")
        print("false activations :", negative.false_events)
        print("FA / hour         :", f"{negative.false_activations_per_hour:.2f}")
        print("peak confidence   :", f"{negative.max_negative_score:.3f}")
        print()
        for name, count in sorted(negative.false_by_category.items(), key=lambda i: -i[1]):
            print(f"  {name:<22} {count}")
        print()
        for item in streaming_evaluation.worst_clips():
            if item.metrics.false_events:
                print(f"  {item.file:<32} {item.metrics.false_events} at "
                      + ", ".join(f"{t:.1f}s" for t in item.false_times[:8]))
    else:
        print("no negative-only recordings. FA/hour is measured on the sessions alone, which")
        print("understates it: real listening is mostly non-target audio. Record an hour of TV.")


## 6 · Hard negatives, clip by clip

Separately from the streaming numbers, because this one names the problem: which *kind* of
near-miss gets through, and which specific recordings are the most confident false positives.


In [ ]:
from src.streaming_eval import evaluate_negative_clips
from src.dataset import filter_split

negative_records = [
    r for r in filter_split(records, config.evaluation.split) if r.negative_type
]
hard_negative_report = None
if negative_records:
    hard_negative_report = evaluate_negative_clips(
        detector, negative_records, root, threshold=operating_point.activation_threshold
    )
    print(hard_negative_report.summary())
else:
    print("no categorised negatives in this split.")


## 7 · Recommended production settings

What Android should use for **this** target. Parameters calibrated for 006 say nothing about
007, so these travel with the model in `model_metadata.json` rather than living as constants in
Kotlin.


In [ ]:
print("activation threshold :", f"{operating_point.activation_threshold:.2f}")
print("release threshold    :", f"{operating_point.release_threshold:.2f}")
print("min consecutive hits :", operating_point.min_consecutive_hits)
print("release windows      :", operating_point.release_windows)
print("cooldown             :", f"{operating_point.cooldown_ms:.0f} ms")
print("window / hop         :", f"{config.audio.clip_seconds:g} s / {config.streaming.hop_seconds:g} s")
print("smoothing            :", config.streaming.smoothing.mode)
if calibration is not None and not calibration.satisfied:
    print()
    print("!! these are the CONFIGURED values, not calibrated ones - calibration failed.")


# 06 · Export

Everything Android needs for this one dhikr, in `exports/<target>/`:

```
dhikr_007_float32.tflite      the reference variant
dhikr_007_int8.tflite         what ships, when quantisation is verified
model_metadata.json           the contract - see below
labels.txt                    unknown / target, in output order
evaluation.json               clip metrics
streaming_evaluation.json     events, FA/hour
calibration.json              the sweep
frontend_test.wav             front-end parity assets
frontend_expected.npy
frontend_metadata.json
mel_filterbank.json
```

Two things this stage does that a plain conversion does not:

**INT8 is not assumed to be free.** Keras, float32 TFLite and INT8 TFLite are scored on the same
positives, ordinary negatives, hard negatives *and* streaming windows. INT8 is recommended only
when it neither drifts beyond tolerance nor adds false activations per hour — a variant that
counts more than the reference is rejected even if its probabilities look close.

**Android needs no hidden constants.** Window geometry, every front-end parameter, the input and
output quantisation scales and zero points, this target's calibrated detector thresholds, and
the measurements the decision was made on all go into `model_metadata.json`.


In [ ]:
from src.pipeline import StreamingOutcome, export_target

outcome = StreamingOutcome(scored=scored, calibration=calibration, evaluation=streaming_evaluation)
exported = export_target(
    preparation,
    model,
    clip_evaluation=clip_evaluation,
    streaming=outcome,
    architecture=ARCHITECTURE,
)
print()
print(exported["quantization"].summary())


## Front-end parity

The model takes a log-mel tensor, not audio, so the Kotlin front-end has to reproduce this one
*exactly*. When it does not, nothing fails: the app runs and the model returns confident
nonsense, indistinguishable from a bad model.

`frontend_test.wav` is written already conditioned — one window long, level-normalised — so the
device-side check is unambiguous: decode it, run the front-end straight over the samples,
compare against `frontend_expected.npy` element-wise.


In [ ]:
from src.audio import read_wav
from src.parity import PARITY_AUDIO_NAME, compare_parity

samples, _ = read_wav(exported["directory"] / PARITY_AUDIO_NAME)
print(compare_parity(extractor(samples), exported["directory"]).summary())
print()
print("copy to app/src/main/assets/dhikr/%s/:" % config.target.folder)
for path in sorted(exported["directory"].iterdir()):
    if path.is_file():
        print("  %-34s %8.1f KB" % (path.name, path.stat().st_size / 1024.0))


## Readiness

The verdict, from the streaming numbers and the dataset that produced them. An unmeasured
criterion counts as unmeasured, not as a pass — so a target with no streaming evaluation can
never read `READY FOR DEVICE TEST`, however high its clip accuracy is.

And `READY FOR DEVICE TEST` is what it says: a licence to try it on a phone. On-device latency,
microphone response and real rooms are still unmeasured here.


In [ ]:
from src.pipeline import TargetRun, readiness_for, write_run_report

run = TargetRun(
    target_id=preparation.target_id,
    target_text=preparation.target_text,
    preparation=preparation,
    clip_evaluation=clip_evaluation,
    streaming=outcome,
    export_dir=exported["directory"],
)
run.readiness = readiness_for(
    run, quantization=exported["quantization"], hard_negatives=hard_negative_report
)
print(run.readiness.summary())
print()
print("run report:", write_run_report(run, paths.reports_path))


# 07 · Architecture comparison (optional)

Single-target detection is a far easier problem than 10-way classification, so a much smaller
network is worth measuring before assuming the big one is needed.

Each architecture is trained on the **same split**, with the same augmentation, optimiser, seed
and epoch budget — `model_presets` overrides only the values that make it that architecture — so
a difference is the architecture and not a hyperparameter.

**Do not pick the winner on accuracy.** The order that matters:

1. false activations / hour
2. event precision
3. event recall
4. hard-negative rejection
5. latency
6. size

and the largest model is not automatically the best one: the smallest that meets the release
criteria is what should ship.


In [ ]:
ARCHITECTURES = ["ds_cnn_tiny", "ds_cnn", "tc_resnet8"]
RUN_COMPARISON = False    # set True; this trains one model per architecture


In [ ]:
import math

from src.export import benchmark_tflite
from src.streaming import StreamingDetector
from src.streaming_eval import calibrate_threshold, evaluate_timelines, score_clips

comparison_rows = []
if RUN_COMPARISON:
    for architecture in ARCHITECTURES:
        print(f"\n{'=' * 60}\n{architecture}\n{'=' * 60}")
        variant_trainer, variant_artifacts = train_target(
            preparation,
            run_name=f"cmp_{config.target.folder}_{architecture}",
            fresh=True,
            architecture=architecture,
            verbose=0,
        )
        variant_model = variant_trainer.load_best()
        variant_config = config.with_overrides({"model.name": architecture})

        variant_clip = evaluate_target(preparation, variant_model)
        variant_detector = StreamingDetector.from_config(variant_config, variant_model)
        variant_scored = score_clips(
            variant_detector, clips, streaming_audio_root(config), target_folder=config.target.folder
        )

        point = config.streaming.detector
        variant_calibration = None
        if variant_scored and config.calibration.enabled:
            variant_calibration = calibrate_threshold(
                variant_scored, config.calibration, point, config.streaming.match_tolerance_seconds
            )
            if variant_calibration.satisfied:
                point = detector_with_threshold(
                    point, variant_calibration.activation,
                    release=variant_calibration.release,
                    release_ratio=config.calibration.release_ratio,
                )
        variant_events = (
            evaluate_timelines(variant_scored, point, config.streaming.match_tolerance_seconds).metrics
            if variant_scored else None
        )

        # Each candidate writes to its own folder: without output_dir they would
        # overwrite exports/<target>/ in turn and the last one trained would
        # silently become what the app loads.
        variant_export = export_target(
            preparation, variant_model,
            clip_evaluation=variant_clip,
            streaming=StreamingOutcome(variant_scored, variant_calibration, None),
            architecture=architecture,
            output_dir=config.target_export_path() / "compare" / architecture,
        )
        int8_path = variant_export["variants"].get("int8") or next(iter(variant_export["variants"].values()))
        benchmark = benchmark_tflite(int8_path, name=architecture, runs=50, warmup=5)

        comparison_rows.append({
            "architecture": architecture,
            "parameters": int(variant_model.count_params()),
            "int8_kb": round(int8_path.stat().st_size / 1024.0, 1),
            "clip_precision": variant_clip.precision,
            "clip_recall": variant_clip.recall,
            "hard_negative_fp": variant_clip.by_negative_type().get("hard_negative", {}).get("false_positive_rate"),
            "event_precision": variant_events.precision if variant_events else None,
            "event_recall": variant_events.recall if variant_events else None,
            "event_f1": variant_events.f1 if variant_events else None,
            "fa_per_hour": variant_events.false_activations_per_hour if variant_events else None,
            "latency_ms": round(benchmark.mean_latency_ms, 2),
            "activation": point.activation_threshold,
        })
else:
    print("set RUN_COMPARISON = True to train one model per architecture.")


In [ ]:
if comparison_rows:
    table = pd.DataFrame(comparison_rows).set_index("architecture")
    display(table.round(4))
    display(viz.plot_architecture_comparison(comparison_rows))

    usable = [
        row for row in comparison_rows
        if row["fa_per_hour"] is not None
        and row["fa_per_hour"] <= config.readiness.max_false_activations_per_hour
        and (row["event_recall"] or 0) >= config.readiness.min_event_recall
        and (row["event_precision"] or 0) >= config.readiness.min_event_precision
    ]
    print()
    if usable:
        # Smallest model that meets the release criteria - not the most accurate one.
        best = min(usable, key=lambda row: row["int8_kb"])
        print("RECOMMENDED:", best["architecture"])
        print(f"  {best['int8_kb']:.1f} KB INT8 | FA/h {best['fa_per_hour']:.2f} | "
              f"event P {best['event_precision']:.1%} R {best['event_recall']:.1%} | "
              f"{best['latency_ms']:.2f} ms")
        print()
        print("Smallest architecture meeting the release criteria. A larger model that scores")
        print("a point higher on clip accuracy buys nothing the user can see.")
    else:
        print("NONE of the architectures met the release criteria. That is a data result, not")
        print("an architecture result - a bigger network will not fix false activations that")
        print("come from never having seen a near-miss of this phrase.")


# 08 · Experiment — one model per dhikr? (historical)

**This question is settled: single-target is the production architecture.** The section is kept
because it is the measurement that settled it, and because it still answers a question the new
pipeline does not — how a *committee* of binary detectors compares with a multi-class model at
telling the nested phrases apart (`سبحان الله` ⊂ `سبحان الله وبحمده` ⊂
`سبحان الله العظيم وبحمده`).

It runs against a **legacy multi-class manifest** (`processed/manifest.csv`, written with
`target.phrase_id: null`), not against the single-target manifest the stages above produce. Skip
it unless you are re-running the comparison; nothing here touches the exported model.

It is not cheap: one training run per phrase on top of the multi-class one.

**What gets compared.** Three questions, because they can disagree:

| Question | Metric | Why it matters |
|---|---|---|
| Can it detect *this* phrase? | ROC-AUC / average precision per phrase | Threshold-free, so neither side wins on tuning |
| Can it name the *right* phrase? | accuracy on phrase clips | Where nested phrases decide it |
| Can it stay quiet on non-dhikr? | accept rate on `unknown` clips | A committee of binaries has no `unknown` output, only a threshold |

**What is held fixed** so the result is about the approach and not the setup: the same manifest
and splits, the same architecture, the same augmentation, the same optimiser, the same seed and
the same epoch budget. The evaluation dataset is built once and every model runs over those same
tensors.

Note that the committee here predicts by `argmax` over the per-model positive scores. That is
*not* what the app does: Android loads exactly one model and reads one score against a
calibrated threshold, so the discrimination column below is a question about this experiment's
deployment shape, not about production.


## 1 · Load the manifest, the trained model and the front-end

In [ ]:
from src.augmentation import NoiseBank, WaveformAugmentor
from src.dataset import class_names_from_manifest, filter_split, load_manifest
from src.experiments import phrase_labels
from src.features import FeatureStats, LogMelExtractor
from src.trainer import load_trained_model

RUN_NAME = config.model.name

paths = config.paths
records = load_manifest(paths.manifest_path)
class_names = class_names_from_manifest(records)

checkpoint = paths.checkpoints_path / RUN_NAME / "best_model.keras"
multiclass_model = load_trained_model(checkpoint)

stats_path = paths.processed_path / "feature_stats.json"
stats = FeatureStats.load(stats_path) if (
    config.features.normalize == "global" and stats_path.is_file()
) else None
extractor = LogMelExtractor(config.features, config.audio.sample_rate, stats=stats)

# The binary models must be trained with the same augmentation the multi-class
# model got, or the comparison measures the augmentation instead of the approach.
noise_bank = NoiseBank.load(
    paths.noise_path,
    config.audio.sample_rate,
    allow_synthetic=config.augmentation.background_noise.synthetic_when_missing,
)
augmentor = WaveformAugmentor(config.augmentation, config.audio, noise_bank)

targets = phrase_labels(class_names, paths.unknown_class)
eval_records = filter_split(records, config.evaluation.split) or filter_split(records, "val")

print("checkpoint :", checkpoint)
print("phrases    : %d %s" % (len(targets), targets))
print("eval clips :", len(eval_records))
print()
print("this will train %d model(s) of %d epochs each"
      % (len(targets), config.training.epochs))
if len(targets) < 2:
    print()
    print("!! only one phrase is in the vocabulary, so there is nothing to tell apart.")
    print("   The per-phrase detection numbers still work; the discrimination test")
    print("   will be skipped. Widen classes.include_phrases for the full comparison.")

## 2 · Train one detector per phrase and compare

Each run is written under `checkpoints/ovr_<phrase>/`, separate from the
multi-class run, and starts from scratch (`fresh=True`) so a detector left over
from a previous experiment cannot bleed into this one.

`EPOCHS = None` gives every binary model the same budget the multi-class model
had, which is what keeps the comparison fair. Shorten it only to rehearse the
mechanics — a shortened run tells you the code works, not which approach wins.

In [ ]:
from src.experiments import compare_one_vs_rest

EPOCHS = None  # None = the same budget as the multi-class run (keep it that way)

report, detectors = compare_one_vs_rest(
    config,
    records,
    multiclass_model,
    extractor,
    augmentor=augmentor,
    phrases=targets,
    epochs=EPOCHS,
    fresh=True,
    verbose=0,
    progress=lambda label, position, total: print(
        "[%d/%d] training one-vs-rest detector for %s ..." % (position, total, label)
    ),
)

print()
print(report.summary())

## 3 · Per-phrase detection

One row per phrase. Both columns answer the same question — how well the
approach separates that phrase from everything else — so the delta is the whole
story. Positive delta means the specialised binary model won.

These are threshold-free (AUC and average precision), so a difference here is a
difference in what the model learned, not in how it was tuned. Average precision
is the one to trust when a phrase has few clips: it is the more honest of the two
under heavy imbalance.

In [ ]:
display(report.to_dataframe())

print("mean AUC — multi-class : %.4f" % report.mean_auc_multiclass)
print("mean AUC — one-vs-rest : %.4f" % report.mean_auc_binary)
print()
for item in report.per_phrase:
    print("%-10s %3d clips | binary trained %d epochs, best val %s"
          % (item.label, item.support, item.binary_epochs,
             "n/a" if item.binary_val_accuracy is None
             else "%.4f" % item.binary_val_accuracy))

## 4 · Telling the phrases apart

Restricted to clips that really are a phrase, and to the phrase columns on both
sides — so `unknown` cannot absorb a mistake for the multi-class model, and the
committee is not asked a question it has no output for.

This is the test the nested prefixes decide. A binary detector for
`سبحان الله وبحمده` is never shown that `سبحان الله العظيم وبحمده` is a
*different* phrase; the multi-class model is trained on exactly that boundary. If
one-vs-rest is going to lose anywhere, it loses here — look at the confusion
pairs, not just the accuracy.

In [ ]:
import pandas as pd

if report.discrimination is None:
    print("skipped — fewer than two phrases in the vocabulary")
else:
    discrimination = report.discrimination
    intervals = discrimination.intervals()
    display(pd.DataFrame([
        {
            "approach": "multi-class (one softmax)",
            "correct": discrimination.multiclass_correct,
            "clips": discrimination.num_clips,
            "accuracy": round(discrimination.multiclass_accuracy, 4),
            "95% CI": "%.3f–%.3f" % intervals["multiclass"],
        },
        {
            "approach": "committee of %d binaries" % discrimination.num_phrases,
            "correct": discrimination.committee_correct,
            "clips": discrimination.num_clips,
            "accuracy": round(discrimination.committee_accuracy, 4),
            "95% CI": "%.3f–%.3f" % intervals["committee"],
        },
    ]))

    for title, pairs in (
        ("multi-class", discrimination.multiclass_confusions),
        ("committee", discrimination.committee_confusions),
    ):
        print()
        print("%s confusions:" % title)
        if not pairs:
            print("  none")
        for true_label, predicted_label, count in pairs:
            print("  %s heard as %s: %d" % (true_label, predicted_label, count))

## 5 · Staying quiet on non-dhikr audio

Both sides are read by the same rule: a clip is *accepted* when the highest
phrase score clears the threshold. The multi-class model's `unknown` output is
deliberately not consulted, so neither approach is judged by a mechanism the
other lacks.

Read the two numbers per approach as a pair. A false-accept rate alone flatters
whichever model fires less often — the recall column is what makes it a
comparison. This is one operating point and the two approaches produce
differently-calibrated scores, so treat it as a sanity check rather than the
verdict.

In [ ]:
if report.rejection is None:
    print("skipped — the vocabulary has no `unknown` class.")
    print("Train with classes.include_unknown and an `unknown` dataset folder to")
    print("measure open-set behaviour; without it neither approach can say")
    print("\"that was not a dhikr\".")
else:
    rejection = report.rejection
    display(pd.DataFrame([
        {
            "approach": "multi-class",
            "fires on unknown": "%d/%d" % (rejection.multiclass_unknown_accepted,
                                           rejection.unknown_clips),
            "false accept": round(rejection.multiclass_false_accept, 4),
            "accepts real phrases": round(rejection.multiclass_recall, 4),
        },
        {
            "approach": "committee",
            "fires on unknown": "%d/%d" % (rejection.committee_unknown_accepted,
                                           rejection.unknown_clips),
            "false accept": round(rejection.committee_false_accept, 4),
            "accepts real phrases": round(rejection.committee_recall, 4),
        },
    ]))
    print("threshold:", rejection.threshold)

## 6 · Save the comparison

Written next to the other reports. Keep it: the next time the "one model per
dhikr" question comes up, this file is the answer for the dataset you had on the
day, and re-running it after the dataset grows is how you find out whether the
answer changed.

In [ ]:
written = report.save(paths.reports_path)
print("saved:", written)
print()
for note in report.verdict():
    print("!!", note)
    print()

print("checkpoints left behind (delete them once you are done — they are not")
print("shipped and each one is a full model):")
for detector in detectors:
    print("  ", paths.checkpoints_path / detector.run_name)